Лычаный НА N33471

Задача 2 "Экономические города"

Импортируем библиотеки.

In [1]:
import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
matplotlib.style.use('ggplot')
%matplotlib inline

from sklearn.cluster import DBSCAN
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

Маунтим Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Открываем датасет из гугл диска

In [3]:
df = pd.read_csv('/content/drive/MyDrive/itmo/3/теория алгоритмов/lab_3/Econom_Cities_data.csv', sep=";", index_col='City', decimal=',')
df.head()

,Work,Price,Salary
City,,,
Amsterdam,1714,65.6,49.0
Athens,1792,53.8,30.4
Bogota,2152,37.9,11.5
Bombay,2052,30.3,5.3
Brussels,1708,73.8,50.5


Размаунтим Google Drive

In [4]:
drive.flush_and_unmount()

Удаляем выбросы

In [5]:
df.drop(df.index[[6,17]],inplace = True)

Создадим копию датасета

In [6]:
df_2 = df.copy()

Стандартизируем данные

In [7]:
from sklearn import preprocessing

In [8]:
stand_1 = preprocessing.StandardScaler()
stand_1.fit(df_2)
X = stand_1.transform(df_2)
X = pd.DataFrame(X, index=df.index, columns=df.columns)

In [9]:
dbscan = DBSCAN(eps=0.5, metric='euclidean', min_samples=5)
dbscan.fit(X)
dbscan.labels_

array([ 0, -1, -1, -1,  0, -1, -1, -1, -1,  0,  0, -1, -1, -1, -1, -1, -1,
       -1, -1, -1,  0, -1, -1, -1, -1, -1,  0,  0, -1, -1, -1, -1, -1,  0,
       -1, -1, -1, -1, -1,  0, -1, -1, -1, -1,  0, -1])

Cоздадим таблицу частот

In [10]:
unique, counts = np.unique(dbscan.labels_, return_counts=True)
print(np.asarray((unique, counts)).T)

[[-1 36]
 [ 0 10]]


In [11]:
df_2['dbscan'] = dbscan.labels_
df_2['dbscan'].sort_values()

City
Luxembourg       -1
Manila           -1
Mexico_City      -1
Nairobi          -1
New_York         -1
Nicosia          -1
Oslo             -1
Panama           -1
Madrid           -1
Rio_de_Janeiro   -1
Seoul            -1
Singpore         -1
Stockholm        -1
Taipei           -1
Tel_Aviv         -1
Tokyo            -1
Toronto          -1
San_Paulo        -1
Los_Angeles      -1
Zurich           -1
Helsinki         -1
Chicago          -1
Copenhagen       -1
Lisbon           -1
Frankfurt        -1
Geneva           -1
Caracas          -1
Hong_Kong        -1
Buenos_Aires     -1
Bogota           -1
Houston          -1
Johannesburg     -1
Kuala_Lumpur     -1
Athens           -1
Lagos            -1
Bombay           -1
Brussels          0
Sydney            0
London            0
Dusseldorf        0
Paris             0
Montreal          0
Milan             0
Vienna            0
Dublin            0
Amsterdam         0
Name: dbscan, dtype: int64

Данные результат нам не подходит, так как всего 2 кластера

In [12]:
dbscan = DBSCAN(eps=0.4, metric='euclidean', min_samples=2)
dbscan.fit(X)
dbscan.labels_

array([ 0,  1,  2, -1,  0, -1,  3,  4, -1,  5,  0,  0, -1, -1, -1, -1, -1,
        2,  6, -1,  5, -1, -1, -1, -1,  7,  5,  4,  7, -1,  1, -1, -1,  5,
        6, -1,  1,  3, -1,  0, -1, -1, -1,  4,  5, -1])

In [13]:
unique, counts = np.unique(dbscan.labels_, return_counts=True)
print(np.asarray((unique, counts)).T)

[[-1 22]
 [ 0  5]
 [ 1  3]
 [ 2  2]
 [ 3  2]
 [ 4  3]
 [ 5  5]
 [ 6  2]
 [ 7  2]]


Мы получили подходяющую модель, так как число кластеров сопоставимо с результатми прошлой работы
Один кластер на столицы, второй на города с низким и средним уровнем жизни. Третий - города Швейцарии. Четвертый -  скандинавские города. Пятый Гонконг как отдельное гос-во почти. Шестой - Стокгольм отделен от Скандинавских стран(странно). Седьмой - Тайбэ́й, столица Китайской Республики, тоже с экономикой как у гос-ва небольшого, восьмой - Токио где вся экономика Японии почти сконцентрирована. 

Вывод: мы получили также 7 кластеров, как и по результам прошлой работы с методом k-means. Правда, распределения по городам немного другие
Таким образом, DBSCAN интересный метод, но работать с ним немного сложнее, так как приходится подбирать параметры в слепую